# Experimentul 6: alegerea pragului pentru recall mai mare (Modelul B, fără reantrenare)

**Întrebare:** VQC-ul simplificat din Experimentul 3 (Modelul B: 4 qubiți, ZZ feature map cu o singură repetare, ansatz cu 3 straturi, 32 de parametri) poate prinde mai multe fraude doar printr-un **prag diferit**, fără reantrenare? Și cât costă asta în alarme false?

## Ce ținem fix
- Doar **Modelul B**, cele 5 seeds `[0, 1, 2, 3, 4]` și **ponderile deja salvate** în `results/result_exp3/weights/` (verificate prin sha256 față de `results/result_exp5/config.json`). Nu reantrenăm nimic și nu modificăm arhitectura.
- Același split (`split_indices.npz`, verificat prin sha256), același `AngleEncoder(4, seed)` fitat pe setul 1:1 din train, același model de zgomot (`noisy_qnode`, `default.mixed`).
- Nivelurile de zgomot: **p = 0, 0.01, 0.05**.

## Doar validation, testul nu se atinge
- Pragurile se aleg **doar** din scorurile și etichetele de validation. Scorurile de validation se **recalculează** din ponderi (nu citim fișierele `.npz` din Exp. 5, care conțin și scorurile de test).
- Nu se încarcă etichetele, features-urile sau scorurile testului; nu se rulează nicio evaluare pe test.
- Consecință: toate metricile de mai jos sunt **pe validation, la un prag ales pe același validation**, deci optimiste. Cât din câștigul de recall se păstrează pe date noi nu se poate spune din acest experiment.

## Strategiile comparate (pe validation, la fiecare seed și p)
| Cod | Regulă | Fezabilitate |
|---|---|---|
| `F1-max` | pragul actual: maxim F1 (`choose_threshold`, identic cu Exp. 3 / Exp. 5) | mereu |
| `recall ≥ 75%` | cel mai mare prag cu recall ≥ 0.75 (adică cele mai puține alarme cu această țintă) | se verifică, nu se presupune |
| `recall ≥ 80%` | cel mai mare prag cu recall ≥ 0.80 | se verifică, nu se presupune |
| `max recall, precision ≥ 60%` | recall maxim printre pragurile cu precision ≥ 0.60 (la egalitate, pragul mai mare) | se verifică, nu se presupune |
| `max recall, FP ≤ 50` | recall maxim printre pragurile cu cel mult 50 de alarme false pe validation (la egalitate, pragul mai mare) | se verifică, nu se presupune |

Validation are 99 de fraude, deci recall ≥ 75% înseamnă TP ≥ 75, iar recall ≥ 80% înseamnă TP ≥ 80.

**AUPRC nu depinde de prag.** AUPRC (average precision) rezumă întreaga curbă precision–recall, deci este identic pentru toate cele cinci strategii la același (seed, p). Pragul alege doar un punct pe curbă: mută compromisul între precision și recall, dar nu face modelul mai bun sau mai slab la ordonarea tranzacțiilor.

## 1. Configurație și imports

**Seeds.** `set_seed` fixează numpy și torch (ca în Experimentele 1–5). Scorarea nu are nimic aleator: simulatoarele PennyLane rulează cu `shots=None` (valori de așteptare exacte, matrici de densitate pentru zgomot), iar `AngleEncoder(4, seed)` folosește `random_state=seed`.

In [ ]:
import hashlib
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_recall_curve
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ---- Modelul B (Exp. 3), neschimbat ----
FEATURE_MAP_REPS = 1
N_QUBITS = 4
ANSATZ_REPS = 3               # (3 + 1) * 4 * 2 = 32 parametri
NATIVE_GATES = {"RX", "RY", "RZ", "Rot", "Hadamard", "CNOT", "CZ"}
SEEDS = [0, 1, 2, 3, 4]
SPLIT_SEED = 42

# ---- Experimentul 6 ----
P_EVAL = [0.0, 0.01, 0.05]
RECALL_TARGETS = [0.75, 0.80]
MIN_PRECISION = 0.60
MAX_FALSE_ALARMS = 50
STRATEGIES = ["F1-max", "recall ≥ 75%", "recall ≥ 80%", "max recall, precision ≥ 60%", "max recall, FP ≤ 50"]

EXPERIMENT_NAME = "threshold_recall"
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
RESULTS_DIR = REPO_ROOT / "results" / "result_exp6"
SCORES_DIR = RESULTS_DIR / "val_scores"
EXP3_DIR = REPO_ROOT / "results" / "result_exp3"     # ponderile Modelului B: doar citit
EXP5_CONFIG = REPO_ROOT / "results" / "result_exp5" / "config.json"   # doar sha256-urile: doar citit
WEIGHTS_DIR = EXP3_DIR / "weights"
SPLIT_PATH = REPO_ROOT / "split_indices.npz"

# culori: paleta categorială implicită, sloturile 1–5 în ordine fixă; formele markerilor dublează culoarea
STRATEGY_STYLE = {
    "F1-max": ("#2a78d6", "o"),
    "recall ≥ 75%": ("#eb6834", "s"),
    "recall ≥ 80%": ("#1baf7a", "D"),
    "max recall, precision ≥ 60%": ("#eda100", "^"),
    "max recall, FP ≤ 50": ("#e87ba4", "v"),
}
CURVE_COLOR = "#8a8f98"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SPLIT_SEED)
print(f"Model B: ZZ ×{FEATURE_MAP_REPS}, {N_QUBITS} qubiți, ansatz {ANSATZ_REPS} straturi   seeds: {SEEDS}   p: {P_EVAL}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Datele: doar train (pentru encoder) și validation

Splitul se verifică prin sha256 față de valoarea înregistrată în Exp. 5. Păstrăm doar rândurile de train (pentru a refita encoderul identic cu Exp. 3) și de validation; indicii, etichetele și features-urile testului sunt aruncați imediat.

In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


exp5_config = json.loads(EXP5_CONFIG.read_text(encoding="utf-8"))
assert sha256(SPLIT_PATH) == exp5_config["split_sha256"], "split_indices.npz diferă de cel folosit în Exp. 3 / Exp. 5"

DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"fișier obligatoriu lipsă: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and df.index.equals(pd.RangeIndex(len(df)))

with np.load(SPLIT_PATH) as split:            # cheia "test" nu se citește
    train_idx, val_idx = split["train"], split["val"]
X_train, y_train = df.drop(columns="Class").loc[train_idx], df["Class"].to_numpy()[train_idx]
X_val, y_val = df.drop(columns="Class").loc[val_idx], df["Class"].to_numpy()[val_idx]
del df                                        # testul nu rămâne în memorie
assert (int(y_train.sum()), int(y_val.sum())) == (295, 99)
N_VAL_FRAUD, N_VAL_LEGIT = int(y_val.sum()), int((y_val == 0).sum())
print(f"train {len(train_idx):,} ({int(y_train.sum())} fraude)   validation {len(val_idx):,} ({N_VAL_FRAUD} fraude, {N_VAL_LEGIT:,} legitime)   test: neîncărcat")

## 3. Funcțiile Modelului B (copiate neschimbate din Exp. 3)

In [ ]:
def undersample(X_part, y_part, seed):
    rng = np.random.default_rng(seed)
    fraud = np.flatnonzero(y_part == 1)
    legit = rng.choice(np.flatnonzero(y_part == 0), size=len(fraud), replace=False)
    keep = rng.permutation(np.concatenate([fraud, legit]))
    return X_part.iloc[keep], y_part[keep]


class AngleEncoder:
    raw_columns = ["Time", "Amount"]

    def __init__(self, n_components, seed):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=seed)
        self.angles = MinMaxScaler(feature_range=(0, np.pi))

    def _scaled(self, frame):
        out = frame.copy()
        out[self.raw_columns] = self.scaler.transform(frame[self.raw_columns])
        return out.to_numpy()

    def fit(self, frame):
        self.scaler.fit(frame[self.raw_columns])
        self.angles.fit(self.pca.fit_transform(self._scaled(frame)))
        return self

    def transform(self, frame):
        return np.clip(self.angles.transform(self.pca.transform(self._scaled(frame))), 0, np.pi)


def feature_map(x):
    """ZZ feature map. x holds N_QUBITS angles in [0, pi], or a batch of them (shape (..., N_QUBITS))."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[..., i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def ansatz(weights):
    """EfficientSU2-style ansatz; weights has shape (ANSATZ_REPS + 1, N_QUBITS, 2)."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def vqc(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


device = qml.device("default.qubit", wires=N_QUBITS)   # shots=None: valori de așteptare exacte
circuit = qml.QNode(vqc, device, interface="torch", diff_method="backprop")


def fraud_probability(x, weights):
    return (1 - circuit(x, weights)) / 2


def predict(Z, weights, model=None, chunk=20000):
    model = model or fraud_probability
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([model(X_t[k:k + chunk], weights) for k in range(0, len(X_t), chunk)]).numpy()


def choose_threshold(y_true, scores):
    """The threshold that maximises F1 on (y_true, scores)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]


def noisy_qnode(circuit_fn, p, n_wires=N_QUBITS):
    """QNode of circuit_fn under the challenge's noise model: native gates, then a depolarizing channel of
    probability p after every gate, on each of its qubits. Differentiable with backprop."""
    noisy_device = qml.noise.insert(qml.device("default.mixed", wires=n_wires), qml.DepolarizingChannel, p, position="all")
    qnode = qml.QNode(circuit_fn, noisy_device, interface="torch", diff_method="backprop")
    return qml.transforms.decompose(qnode, gate_set=NATIVE_GATES)


def noisy_predict(Z, weights, p, circuit_fn=vqc, readout=lambda expval: (1 - expval) / 2, n_wires=N_QUBITS, chunk=2000):
    """Fraud scores of the transactions Z when circuit_fn runs under depolarizing noise p."""
    noisy_circuit = noisy_qnode(circuit_fn, p, n_wires)
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([readout(noisy_circuit(X_t[k:k + chunk], weights)) for k in range(0, len(X_t), chunk)]).numpy()

## 4. Ponderile salvate ale Modelului B

Fiecare `seed_<s>.pt` din `results/result_exp3/weights/` trebuie să aibă sha256-ul înregistrat pentru Modelul B în Exp. 5, iar AUPRC-ul pe validation la p = 0 trebuie să reproducă valoarea salvată la antrenare.

In [ ]:
models = {}
for seed in SEEDS:
    path = WEIGHTS_DIR / f"seed_{seed}.pt"
    assert sha256(path) == exp5_config["weights_sha256"][f"B/seed_{seed}.pt"], f"{path} nu este Modelul B din Exp. 5"
    saved = torch.load(path, weights_only=True)
    X_s, _ = undersample(X_train, y_train, seed)
    Z_val = AngleEncoder(N_QUBITS, seed).fit(X_s).transform(X_val)
    assert np.isclose(average_precision_score(y_val, predict(Z_val, saved["weights"])), saved["best_val_ap"]), f"seed {seed}: AUPRC diferit"
    models[seed] = {"weights": saved["weights"], "Z_val": Z_val}
    print(f"seed {seed}: ponderi Model B încărcate (sha256 OK)   val AUPRC (p = 0) {saved['best_val_ap']:.4f}")
del X_train, y_train

## 5. Scorurile de validation la p = 0, 0.01, 0.05

Scorurile se recalculează din ponderi și se salvează în `results/result_exp6/val_scores/` (doar validation); la o rulare ulterioară se încarcă de acolo. Verificare: pragul F1-max, TP, FP și AUPRC trebuie să coincidă cu `results/result_exp3/val_runs.jsonl`.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SCORES_DIR.mkdir(parents=True, exist_ok=True)
exp3_val = {(r["seed"], r["p"]): r for r in map(json.loads, (EXP3_DIR / "val_runs.jsonl").read_text(encoding="utf-8").splitlines()) if r}

val_scores, scoring_time = {}, {}
for seed in SEEDS:
    m = models[seed]
    for p in P_EVAL:
        path = SCORES_DIR / f"B_seed{seed}_p{p}.npz"
        if path.exists():
            with np.load(path) as f:
                scores, seconds = f["val"], float(f["seconds"])
        else:
            t0 = time.perf_counter()
            scores = predict(m["Z_val"], m["weights"]) if p == 0.0 else noisy_predict(m["Z_val"], m["weights"], p)
            seconds = time.perf_counter() - t0
            np.savez(path, val=scores, seconds=seconds)
        ref = exp3_val[(seed, p)]
        threshold = choose_threshold(y_val, scores)
        tn, fp, fn, tp = confusion_matrix(y_val, (scores >= threshold).astype(int), labels=[0, 1]).ravel()
        assert np.isclose(threshold, ref["threshold"]) and (tp, fp) == (ref["TP"], ref["FP"]), f"seed {seed}, p = {p}: diferă de Exp. 3"
        assert np.isclose(average_precision_score(y_val, scores), ref["AUPRC"]), f"seed {seed}, p = {p}: AUPRC diferă de Exp. 3"
        val_scores[(seed, p)], scoring_time[(seed, p)] = scores, seconds
print(f"✅ {len(val_scores)} seturi de scoruri de validation; pragul F1-max, TP, FP și AUPRC coincid cu Exp. 3")

## 6. Strategiile de prag

Pentru fiecare (seed, p), candidații sunt toate valorile distincte ale scorurilor de validation (tranzacția e marcată fraudă dacă scorul ≥ prag). Pentru fiecare candidat știm TP și FP, deci fiecare strategie este o selecție exactă pe această listă. Dacă nicio valoare nu respectă constrângerea, strategia e marcată **nefezabilă** pentru acel (seed, p) și nu intră în medie.

In [ ]:
def operating_points(y_true, scores):
    """TP and FP for every distinct threshold (prediction: score >= threshold), threshold descending."""
    order = np.argsort(-scores, kind="mergesort")
    s, y = scores[order], y_true[order]
    tp, fp = np.cumsum(y), np.cumsum(1 - y)
    last = np.r_[np.flatnonzero(s[1:] != s[:-1]), len(s) - 1]   # ultimul element din fiecare grup de scoruri egale
    return pd.DataFrame({"threshold": s[last], "TP": tp[last], "FP": fp[last]})


def pick(points, mask, by_recall):
    """Among the points in mask: the highest threshold (fewest alarms), or the highest TP then the highest threshold."""
    cand = points[mask]
    if cand.empty:
        return None
    if by_recall:
        cand = cand[cand["TP"] == cand["TP"].max()]
    return cand["threshold"].max()


def strategy_thresholds(y_true, scores):
    pts = operating_points(y_true, scores)
    n_pos = int(y_true.sum())
    precision = pts["TP"] / (pts["TP"] + pts["FP"])
    out = {"F1-max": choose_threshold(y_true, scores)}
    for target, name in zip(RECALL_TARGETS, ["recall ≥ 75%", "recall ≥ 80%"]):
        min_tp = int(np.ceil(target * n_pos - 1e-9))     # 99 fraude: 75% → 75, 80% → 80
        out[name] = pick(pts, pts["TP"] >= min_tp, by_recall=False)
    out["max recall, precision ≥ 60%"] = pick(pts, precision >= MIN_PRECISION, by_recall=True)
    out["max recall, FP ≤ 50"] = pick(pts, pts["FP"] <= MAX_FALSE_ALARMS, by_recall=True)
    return out


def counts_at(y_true, scores, threshold):
    tn, fp, fn, tp = confusion_matrix(y_true, (scores >= threshold).astype(int), labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn)
    return {"TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn), "precision": precision, "recall": recall,
            "F1": f1_score(y_true, (scores >= threshold).astype(int), zero_division=0)}


rows = []
for seed in SEEDS:
    for p in P_EVAL:
        scores = val_scores[(seed, p)]
        auprc = average_precision_score(y_val, scores)
        for name, thr in strategy_thresholds(y_val, scores).items():
            row = {"p": p, "seed": seed, "strategie": name, "fezabil": thr is not None, "prag": np.nan if thr is None else float(thr), "AUPRC": auprc}
            if thr is not None:
                row.update(counts_at(y_val, scores, thr))
            rows.append(row)
per_seed = pd.DataFrame(rows)
per_seed["alarme totale"] = per_seed["TP"] + per_seed["FP"]

# verificări: constrângerile sunt respectate și AUPRC e același pentru toate strategiile
ok = per_seed[per_seed["fezabil"]]
assert (ok.loc[ok["strategie"] == "recall ≥ 75%", "recall"] >= 0.75).all() and (ok.loc[ok["strategie"] == "recall ≥ 80%", "recall"] >= 0.80).all()
assert (ok.loc[ok["strategie"] == "max recall, precision ≥ 60%", "precision"] >= MIN_PRECISION).all()
assert (ok.loc[ok["strategie"] == "max recall, FP ≤ 50", "FP"] <= MAX_FALSE_ALARMS).all()
assert (per_seed.groupby(["seed", "p"])["AUPRC"].nunique() == 1).all()

# diferențe pereche față de F1-max, pe același (seed, p)
f1_ref = per_seed[per_seed["strategie"] == "F1-max"].set_index(["seed", "p"])
for col in ["TP", "FP", "precision", "recall", "F1"]:
    per_seed[f"Δ{col} vs F1-max"] = per_seed[col].to_numpy() - f1_ref.loc[list(zip(per_seed["seed"], per_seed["p"])), col].to_numpy()

per_seed.to_csv(RESULTS_DIR / "strategies_per_seed.csv", index=False)
print("Fezabilitate (seeds din 5 pentru care strategia există pe validation):")
display(per_seed.pivot_table(index="strategie", columns="p", values="fezabil", aggfunc="sum").loc[STRATEGIES].astype(int))

## 7. Rezultate: medie ± std pe 5 seeds (validation)

Media ± std (ddof = 1) se face doar pe seeds-urile fezabile (coloana `n`). `ΔTP` și `ΔFP` sunt diferențe **pereche** față de pragul F1-max la același seed și p: fraude suplimentare detectate și alarme false suplimentare. `FP / 10k legitime` normalizează alarmele false la 10.000 de tranzacții legitime din validation.

In [ ]:
def mean_std(values, digits=3):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return "nefezabil"
    if len(values) == 1:
        return f"{values[0]:.{digits}f} (1 seed)"
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


summary_rows, numeric_rows = [], []
for p in P_EVAL:
    for name in STRATEGIES:
        g = per_seed[(per_seed["p"] == p) & (per_seed["strategie"] == name)]
        f = g[g["fezabil"]]
        summary_rows.append({
            "p": p, "strategie": name, "n": len(f),
            "TP": mean_std(f["TP"], 1), "FP": mean_std(f["FP"], 1), "FN": mean_std(f["FN"], 1),
            "precision": mean_std(f["precision"]), "recall": mean_std(f["recall"]), "F1": mean_std(f["F1"]),
            "ΔTP vs F1-max": mean_std(f["ΔTP vs F1-max"], 1), "ΔFP vs F1-max": mean_std(f["ΔFP vs F1-max"], 1),
            "FP / 10k legitime": mean_std(f["FP"] / N_VAL_LEGIT * 1e4, 2),
            "AUPRC (nu depinde de prag)": mean_std(g["AUPRC"]),
        })
        numeric_rows.append({"p": p, "strategie": name, "n": len(f),
                             **{f"{c} medie": f[c].mean() for c in ["TP", "FP", "FN", "precision", "recall", "F1", "ΔTP vs F1-max", "ΔFP vs F1-max"]},
                             **{f"{c} std": f[c].std(ddof=1) for c in ["TP", "FP", "FN", "precision", "recall", "F1", "ΔTP vs F1-max", "ΔFP vs F1-max"]}})
summary = pd.DataFrame(summary_rows).set_index(["p", "strategie"])
summary.to_csv(RESULTS_DIR / "strategies_summary.csv")
pd.DataFrame(numeric_rows).to_csv(RESULTS_DIR / "strategies_summary_numeric.csv", index=False)
for p in P_EVAL:
    print(f"p = {p} · Modelul B · validation · medie ± std pe seeds fezabile")
    display(summary.loc[p])

### Compromisul față de pragul F1-max

Pentru fiecare strategie: câte fraude în plus prindem (ΔTP), câte alarme false în plus plătim (ΔFP) și **câte alarme false suplimentare costă fiecare fraudă suplimentară** (Σ ΔFP / Σ ΔTP pe seeds fezabile; nedefinit când ΔTP ≤ 0).

In [ ]:
trade_rows = []
for p in P_EVAL:
    for name in STRATEGIES[1:]:
        f = per_seed[(per_seed["p"] == p) & (per_seed["strategie"] == name) & per_seed["fezabil"]]
        d_tp, d_fp = f["ΔTP vs F1-max"].sum(), f["ΔFP vs F1-max"].sum()
        trade_rows.append({
            "p": p, "strategie": name, "n": len(f),
            "fraude suplimentare (ΔTP)": mean_std(f["ΔTP vs F1-max"], 1),
            "alarme false suplimentare (ΔFP)": mean_std(f["ΔFP vs F1-max"], 1),
            "Δrecall [pp]": mean_std(100 * f["Δrecall vs F1-max"], 1),
            "Δprecision [pp]": mean_std(100 * f["Δprecision vs F1-max"], 1),
            "FP suplimentare / fraudă suplimentară": f"{d_fp / d_tp:.1f}" if d_tp > 0 else "—",
            "seeds cu ΔTP > 0": f"{int((f['ΔTP vs F1-max'] > 0).sum())} / {len(f)}",
        })
tradeoff = pd.DataFrame(trade_rows).set_index(["p", "strategie"])
tradeoff.to_csv(RESULTS_DIR / "tradeoff_vs_f1max.csv")
display(tradeoff)

print("Pragul F1-max (referința), validation:")
display(summary.xs("F1-max", level="strategie")[["TP", "FP", "FN", "precision", "recall", "F1"]])

## 8. Grafice

**Curbele precision–recall** (câte una per seed, gri) cu punctul ales de fiecare strategie. Aria de sub curbă (AUPRC) e fixă; strategiile doar se mută pe curbă. Liniile punctate marchează recall 75% / 80% și precision 60%.

In [ ]:
handles = [plt.Line2D([], [], color=c, marker=mk, linestyle="", markersize=8, markeredgecolor="white", label=n) for n, (c, mk) in STRATEGY_STYLE.items()]

fig, axes = plt.subplots(1, len(P_EVAL), figsize=(16, 4.8), sharey=True)
for ax, p in zip(axes, P_EVAL):
    for seed in SEEDS:
        precision, recall, _ = precision_recall_curve(y_val, val_scores[(seed, p)])
        ax.plot(recall, precision, color=CURVE_COLOR, linewidth=1, alpha=0.7, zorder=1)
    for name, (color, marker) in STRATEGY_STYLE.items():
        f = per_seed[(per_seed["p"] == p) & (per_seed["strategie"] == name) & per_seed["fezabil"]]
        ax.scatter(f["recall"], f["precision"], color=color, marker=marker, s=60, edgecolor="white", linewidth=1.2, zorder=3)
    for r in RECALL_TARGETS:
        ax.axvline(r, color="0.4", linestyle=":", linewidth=1)
    ax.axhline(MIN_PRECISION, color="0.4", linestyle=":", linewidth=1)
    auprc = per_seed[(per_seed["p"] == p) & (per_seed["strategie"] == "F1-max")]["AUPRC"]
    ax.set_title(f"p = {p}   ·   AUPRC {mean_std(auprc)}")
    ax.set_xlabel("recall (validation)")
    ax.set_xlim(0.3, 1.0)
    ax.set_ylim(0, 1.02)
axes[0].set_ylabel("precision (validation)")
fig.legend(handles=handles, loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Modelul B · curbe precision–recall pe validation (gri: câte o curbă per seed) și pragurile alese de fiecare strategie", fontsize=11)
plt.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(RESULTS_DIR / "pr_curves.png", dpi=130, bbox_inches="tight")
plt.show()

**Fraude detectate vs alarme false.** Aceeași informație, în unități concrete: pe axa x numărul de tranzacții legitime marcate (FP, scară logaritmică, ca să se vadă și zona 10–100), pe axa y fraudele prinse (TP, din 99). Cât de plată e curba arată câte alarme false costă fiecare fraudă în plus. Liniile punctate: TP = 75 și 80 (recall 75% / 80%), FP = 50.

In [ ]:
fp_max = per_seed.loc[per_seed["fezabil"], "FP"].max()
x_max = max(4 * MAX_FALSE_ALARMS, 1.15 * fp_max)

fig, axes = plt.subplots(1, len(P_EVAL), figsize=(16, 4.8), sharey=True)
for ax, p in zip(axes, P_EVAL):
    for seed in SEEDS:
        pts = operating_points(y_val, val_scores[(seed, p)])
        pts = pts[pts["FP"] <= x_max]
        ax.step(pts["FP"], pts["TP"], where="post", color=CURVE_COLOR, linewidth=1, alpha=0.7, zorder=1)
    for name, (color, marker) in STRATEGY_STYLE.items():
        f = per_seed[(per_seed["p"] == p) & (per_seed["strategie"] == name) & per_seed["fezabil"]]
        ax.scatter(f["FP"], f["TP"], color=color, marker=marker, s=60, edgecolor="white", linewidth=1.2, zorder=3)
    for r in RECALL_TARGETS:
        ax.axhline(np.ceil(r * N_VAL_FRAUD - 1e-9), color="0.4", linestyle=":", linewidth=1)
    ax.axvline(MAX_FALSE_ALARMS, color="0.4", linestyle=":", linewidth=1)
    ax.set_title(f"p = {p}")
    ax.set_xlabel(f"alarme false FP (din {N_VAL_LEGIT:,} legitime, scară log)")
    ax.set_xscale("log")
    ax.set_xlim(5, x_max)
    ax.set_ylim(40, N_VAL_FRAUD + 2)
axes[0].set_ylabel(f"fraude detectate TP (din {N_VAL_FRAUD})")
fig.legend(handles=handles, loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Modelul B · validation · compromisul fraude detectate vs alarme false (gri: câte o curbă per seed)", fontsize=11)
plt.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(RESULTS_DIR / "tradeoff_frauds_vs_false_alarms.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
config = {
    "experiment": EXPERIMENT_NAME, "model": "B · VQC simplificat (Exp. 3, ZZ ×1)", "weights": "results/result_exp3/weights (sha256 verified vs Exp. 5)",
    "feature_map_reps": FEATURE_MAP_REPS, "ansatz_reps": ANSATZ_REPS, "n_qubits": N_QUBITS, "seeds": SEEDS, "noise_levels": P_EVAL,
    "strategies": {"F1-max": "choose_threshold (max F1 on validation)",
                   "recall ≥ 75%": "highest threshold with TP >= ceil(0.75 * 99)", "recall ≥ 80%": "highest threshold with TP >= ceil(0.80 * 99)",
                   "max recall, precision ≥ 60%": "max TP with precision >= 0.60, ties -> highest threshold",
                   "max recall, FP ≤ 50": "max TP with FP <= 50, ties -> highest threshold"},
    "threshold_selection_set": "validation only", "test_used": False, "retraining": False,
    "noise_model": "noisy_qnode: NATIVE_GATES decomposition, DepolarizingChannel(p) after every gate on every wire, default.mixed",
    "split_sha256": exp5_config["split_sha256"],
    "scoring_time_s": {f"seed{s}_p{p}": t for (s, p), t in scoring_time.items()},
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
}
(RESULTS_DIR / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print("salvat:", sorted(p.name for p in RESULTS_DIR.iterdir()))

## 9. Cum citim rezultatele

- Toate cifrele sunt pe **validation**, cu pragul ales pe același validation: sunt optimiste pentru **toate** strategiile, inclusiv F1-max. Comparația între strategii este corectă (aceleași scoruri, același set), dar nu spune cât din câștig se păstrează pe test.
- **AUPRC este independent de prag** și e identic în toate rândurile pentru același (seed, p). Schimbarea pragului nu îmbunătățește modelul; doar alege alt punct pe aceeași curbă.
- Cu 99 de fraude în validation, o fraudă = 1 punct procentual de recall, deci diferențele mici (1–3 fraude) sunt în zgomotul statistic.
- Această analiză **nu** se numără drept îmbunătățire anti-zgomot pentru Task 2 (alegerea pragului pe validare zgomotoasă e exclusă explicit).
- Testul nu a fost folosit. Dacă echipa decide să adopte o strategie, ea trebuie fixată acum, pe validation, înainte de orice evaluare pe test.